# SMS Spam Detection

**Goal:** Classify SMS as Spam or Ham
**Algorithm:** Naive Bayes + TF-IDF
**Dataset:** [SMS Spam Collection](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [1]:
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
else:
    print('Running locally')

Running locally - ready


## 1. Load Data

In [1]:
path = kagglehub.dataset_download('uciml/sms-spam-collection-dataset')
df = pd.read_csv(f'{path}/spam.csv', encoding='latin-1')
df = df[['v1','v2']].rename(columns={'v1':'label','v2':'message'})
print('Shape:', df.shape)
print(df['label'].value_counts().to_dict())

Shape: (5572, 2)
{'ham': 4825, 'spam': 747}


## 2. Preprocess

In [1]:
import re
def clean(t):
    return re.sub(r'[^a-z0-9\s]', '', t.lower()).strip()
df['clean'] = df['message'].apply(clean)

In [1]:
vec = TfidfVectorizer(max_features=3000, stop_words='english')
X = vec.fit_transform(df['clean']).toarray()
y = df['label'].map({'ham':0,'spam':1})

In [1]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape[0], 'Test:', X_test.shape[0])

Train: 4457, Test: 1115


## 3. Train & Evaluate

In [1]:
model = MultinomialNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Accuracy: %.4f' % accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Ham','Spam']))

Accuracy: 0.9839
              precision    recall  f1-score   support
         Ham       0.99      0.99      0.99       965
        Spam       0.97      0.92      0.94       150
    accuracy                           0.98      1115


## 4. Test

In [1]:
for m in ['Hey how are you?', 'WINNER! Claim your prize now!']:
    v = vec.transform([clean(m)]).toarray()
    p = model.predict(v)[0]
    print(f"'{m}' -> {'HAM' if p==0 else 'SPAM'}")

'Hey how are you?' -> HAM
'WINNER! Claim your prize now!' -> SPAM
